# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Omahcy/FlyRank_Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### 1) The Contract, in Plain Words

1. **What one row means (The Grain):**
   One row represents a single client content asset observed on a single calendar date (`client_hash_id` × `content_hash_id` × `report_date`).

2. **Which table(s) I will use:**
   `fact_content_daily_performance` partition for mid-panel month `month=2025-03` from `FlyRank/internship-warehouse`.

3. **Which time window:**
   The full mid-panel month of March 2025 (`2025-03-01` to `2025-03-31`).

4. **What I predict or rank (Label or Proxy):**
   A binary directional indicator (`label_high_engagement`), predicting whether an asset achieves top-tier organic engagement ($\ge 5$ engaged sessions) over the evaluation window.

5. **One thing I deliberately exclude:**
   Real-time downstream engagement duration (`ga4_total_engagement_sec`) and scroll events recorded after the decision moment ($T_0$), as they represent unobserved future outcomes.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field Categorization

* **Context / Primary Keys (Grain identifiers):**
  * `client_hash_id`: Unique identifier for the client domain.
  * `content_hash_id`: Unique identifier for the individual content asset/URL.
  * `report_date`: Observation calendar date.
  * `gsc_data_available`: Boolean flag confirming GSC reporting readiness.
  * `ga4_data_available`: Boolean flag confirming GA4 tracking availability.

* **Features (Knowable at the decision moment $T_0$):**
  * `gsc_impressions`: Baseline organic search impressions aggregated over the historical window.
  * `gsc_avg_position`: Historical mean rank position in organic search results.
  * `sessions_organic`: Historical organic sessions logged prior to optimization action.
  * `sessions_direct`: Settled direct traffic baseline volume.
  * `sessions_ai`: AI-mediated referral sessions (ChatGPT, Perplexity, Gemini) recorded in prior periods.

* **Label / Target (Downstream performance to predict):**
  * `label_high_engagement`: Binary proxy indicator (`ga4_engaged_sessions >= 5`), representing top-tier content engagement.

* **Excluded (Deliberately dropped):**
  * `ga4_total_engagement_sec`: **Why excluded:** Total engagement time accumulates continuously after the session begins; using intraday or downstream engagement duration leaks future user activity that is unobserved at decision time ($T_0$).
  * `scroll_events`: **Why excluded:** Deep-page scroll depth is a downstream behavioral response recorded post-arrival, which is unobservable when planning content distribution.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# Claim 1 Verification: Grain Proof
# Proves that (client_hash_id, content_hash_id, report_date) is strictly unique with 0 duplicate rows
q1_grain = f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM read_parquet('{DATA_PATH}')
GROUP BY client_hash_id, content_hash_id, report_date
HAVING COUNT(*) > 1;
"""

grain_check = con.execute(q1_grain).df()
print(f"Duplicate rows found at grain: {len(grain_check)} (0 confirms grain holds)")
display(grain_check)

In [ ]:
# Claim 2 Verification: Date Window & Row Counts
# Proves the mid-panel slice coverage (March 2025) and client/content breadth
q2_bounds = f"""
SELECT
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_content_assets,
    COUNT(DISTINCT client_hash_id) AS unique_clients
FROM read_parquet('{DATA_PATH}');
"""

bounds_df = con.execute(q2_bounds).df()
display(bounds_df)

In [ ]:
# Claim 3 Verification: Data Availability & Clean Filter
# Uses IS TRUE filter on boolean flags to confirm surviving rows and missingness
q3_availability = f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
    ROUND(SUM(CASE WHEN gsc_data_available IS TRUE THEN 1.0 ELSE 0.0 END) * 100.0 / COUNT(*), 2) AS gsc_pct_available,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
    ROUND(SUM(CASE WHEN ga4_data_available IS TRUE THEN 1.0 ELSE 0.0 END) * 100.0 / COUNT(*), 2) AS ga4_pct_available,
    SUM(CASE WHEN (gsc_data_available IS TRUE AND ga4_data_available IS TRUE) THEN 1 ELSE 0 END) AS joint_available_rows,
    ROUND(SUM(CASE WHEN (gsc_data_available IS TRUE AND ga4_data_available IS TRUE) THEN 1.0 ELSE 0.0 END) * 100.0 / COUNT(*), 2) AS joint_pct_surviving
FROM read_parquet('{DATA_PATH}');
"""

avail_df = con.execute(q3_availability).df()
display(avail_df)

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### 4. Data Limits: What This Slice Can Never Tell You

1. **Unbalanced History (Cold-Start Confounding):**
   * Newly published content assets and recently onboarded clients lack uniform historical depth.
   * A zero in historical impression or session counts conflates **true zero demand** with **lack of observational tenure**. The data cannot distinguish between a piece of content that Google actively ignored versus one that was indexed only 48 hours ago.

2. **GSC-Only Early Rows (Asynchronous Instrumentation):**
   * Clients frequently configure Search Console before completing full GA4 tag setup, and daily sync pipelines face a 48–72 hour settlement lag in Search Console compared to real-time GA4 sessions.
   * For rows where `gsc_data_available IS TRUE` but `ga4_data_available IS FALSE`, this table can never reveal post-click behavioral quality (session duration, bounce, or conversions). It captures search visibility in a vacuum.

3. **Window Overlaps and Edge Truncation:**
   * Daily reporting grain creates severe autocorrelation: consecutive days share rolling historical contexts, meaning row-to-row observations are not independent.
   * Near the month boundary (March 25–31, 2025), downstream evaluation windows (e.g., next-7-day performance) extend beyond the isolated mid-panel partition, creating right-censored outcomes unless future partitions are explicitly queried.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.